# Day 37：阶段项目二(下)：企业级知识库问答系统——多轮改写、引用高亮、SSE流式与全链路联调（Jupyter Notebook 实战手册）

> **所属阶段**：第 3 阶段 · 第三阶段：LangChain 与企业级 RAG 开发 ｜ **第 6 周**：第6周：阶段项目二——企业级知识库问答系统实战与答辩  
> **GitLab 工单**：`STARMIND-037` ｜ **开发分支**：`feat/day37-project2-multiturn-sse-e2e`  
> **配套 HTML5 课件**：`courseware/day37/index.html` ｜ **独立版本快照**：`versions/day37/`

---

## 🎙️ 老陈开场旁白

早啊各位！昨天前端同学提的这三个需求——**多轮上下文改写、引用证据高亮、SSE 流式输出**——正是把一个「能用的后端接口」打磨成「员工每天离不开的 AI 产品」的临门一脚！
今天咱们把项目二（下）彻底封板：
1. **多会话管理器（SessionManager）**：为每个员工/对话窗口维护独立 `session_id` 的滑动窗口历史记录，互不串线；
2. **多轮指代消解改写器（MultiTurnQueryRewriter）**：员工第一轮问「北京出差住宿上限是多少？」，第二轮追问「那餐饮补助呢？」或「它需要提前几天OA申请？」，系统自动把省略的主语补全为可独立检索的完整 Query；
3. **引用溯源高亮器（CitationHighlighter）**：自动定位命中的关键证据词并包上 `<mark>...</mark>` 标签，前端直接渲染出荧光笔高亮卡片；
4. **SSE（Server-Sent Events）流式接口**：基于 FastAPI `StreamingResponse(media_type='text/event-stream')` 按协议顺序推送 `event: meta -> event: citation -> event: token -> event: done`，并用 `TestClient` 跑通全链路端到端联调！

### 承上启下
- **昨日基线**：昨天（Day 36）我们完成了项目二（上）：多格式文档解析入库、ACL 权限过滤、混合检索+Rerank 引擎和 FastAPI 基础接口。但前端联调时立刻提了三个最影响用户体验的硬需求：「1. 员工连续追问『那需要提前几天申请？』系统忘了上一轮在聊年假；2. 引用片段是一坨纯文本，用户找不到重点词；3. 等整段答案拼完才返回太慢，必须上 ChatGPT 同款 SSE 流式打字机！」
- **明日预告**：明天（Day 38）我们将迎来第三阶段收官大战——**项目二答辩与代码评审**：准备 10 分钟技术答辩架构讲稿、执行 GitLab Code Review 评分表、并对项目二进行并发压测瓶颈定位与重构优化实战！


In [ ]:
# 环境自检与路径初始化：确保可以无缝导入当日版本模块并支持异步协程运行
import os
import sys
from pathlib import Path

# 允许在 Jupyter 已有事件循环内安全调用 asyncio.run()
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'versions').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DAY_DIR = PROJECT_ROOT / 'versions' / 'day37'
for _p in (DAY_DIR, DAY_DIR / 'src', PROJECT_ROOT):
    if _p.exists() and str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
os.chdir(DAY_DIR)
__file__ = str(DAY_DIR / 'src' / 'notebook_entry.py')

print(f'✅ Python 版本: {sys.version.split()[0]}')
print(f'✅ 项目根目录: {PROJECT_ROOT}')
print(f'✅ 当日版本目录: {DAY_DIR}')


## 📋 企业需求背景与验收目标（STARMIND-037）

项目二后端单轮问答 API 已就绪，现需补齐企业级聊天产品的核心交互闭环：多会话并发隔离、多轮追问上下文改写、引用片段关键词高亮溯源，以及降低用户首字等待焦虑的 SSE 流式传输。

**今日迭代目标：**
- 实现 `SessionManager`：支持多会话创建、列表、滑动窗口历史查询、删除与状态隔离。
- 实现 `MultiTurnQueryRewriter`：结合会话历史自动识别指代词与短追问，输出带上下文锚点的规范化检索 Query。
- 实现 `CitationHighlighter`：自动提取查询关键词并在引用片段中插入 `<mark>...</mark>` 高亮标签。
- 实现 FastAPI SSE 流式端点 `POST /api/v1/chat/stream`，按标准 SSE 帧协议依次推送 `meta`、`citation`、`token`、`done` 事件。

**验收标准：**
- 两个不同 `session_id` 的并发对话历史 100% 隔离，互不干扰。
- 在「年假规定」之后的追问「它需要提前几天申请？」能被正确改写包含「年假」并精准命中年假审批条款。
- 返回的引用片段包含有效的 `<mark>...</mark>` 高亮标签且不破坏原文完整性。
- SSE 接口响应头包含 `text/event-stream`，客户端解析全部 `token` 帧后可无损还原完整回答，且本轮对话自动持久化至会话历史。

---

## 📚 课堂核心技术笔记

### 1. 1. 为什么大模型流式问答首选 SSE 而不是 WebSocket？

AI 问答天然是「客户端发一次请求 -> 服务端单向流式吐字」的半双工模式：
- **SSE 优势**：纯标准 HTTP/1.1 或 HTTP/2 协议，天然穿透企业防火墙和代理，支持 `POST` 携带 JSON Body 与 JWT Header，服务端无状态扩展成本远低于 WebSocket；
- **WebSocket 适用场景**：需要实时全双工语音打断（如实时语音助手）或多人协同白板时才需要上 WebSocket。

> **⚠️ 老陈避坑**：踩坑提醒：很多同学在本地测 SSE 打字机丝滑流畅，一上测试环境经过 Nginx 反向代理就变成「卡住 5 秒然后瞬间全吐出来」！原因就是 Nginx 默认开启了 `proxy_buffering on`。务必在 FastAPI 响应头加上 `X-Accel-Buffering: no` 并在 Nginx 配 `proxy_buffering off;`！

### 2. 2. 引用高亮防 XSS 安全红线

前端渲染后端返回的 `<mark>关键词</mark>` 时通常需要 `v-html` 或 `dangerouslySetInnerHTML`。因此后端在插入 `<mark>` 标签**之前**，必须先对原始文档内容执行 `html.escape()` 转义掉 `<script>` 等恶意标签，确保只有后端生成的 `<mark>` 标签是真实 HTML，彻底杜绝存储型 XSS 漏洞！

> **⚠️ 老陈避坑**：踩坑提醒：先 `html.escape(snippet)`，再对转义后的安全文本做 `<mark>` 包裹，顺序千万别写反！


## 💻 实操步骤 1：步骤一：实现多会话管理、多轮指代消解改写、安全高亮与 SSE 流式服务引擎

- **目标文件**：`versions/day37/src/day37_session_highlight.py`
- **老陈代码旁白**：咱们把项目二（下）的核心逻辑全部封装进 `day37_session_highlight.py`：包含多会话隔离的 `SessionManager`、指代消解 `MultiTurnQueryRewriter`、带 XSS 防护的 `CitationHighlighter`，以及同时支持同步问答和 SSE 流式事件生成的 `StreamingKBService`。


In [ ]:
"""
Day 37 - 步骤一：阶段项目二(下) 多会话管理、多轮查询改写、安全引用高亮与 SSE 流式引擎
"""

from __future__ import annotations

import html
import json
import re
import time
import uuid
from dataclasses import dataclass, field
from typing import Any, Dict, Generator, List, Optional, Sequence, Tuple


@dataclass
class ChatTurn:
    role: str  # "user" | "assistant"
    content: str
    rewritten_query: Optional[str] = None


@dataclass
class HighlightedCitation:
    index: int
    doc_id: str
    source: str
    section: str
    raw_snippet: str
    highlighted_snippet: str
    score: float


class SessionManager:
    """多会话对话历史管理器：支持按 session_id 隔离与滑动窗口自动裁剪。"""

    def __init__(self, max_turns: int = 6) -> None:
        self.max_turns = max_turns
        self._sessions: Dict[str, Dict[str, Any]] = {}

    def create_session(self, title: str = "新对话", department: str = "public") -> str:
        sid = f"sess_{uuid.uuid4().hex[:10]}"
        self._sessions[sid] = {
            "session_id": sid,
            "title": title,
            "department": department,
            "history": [],
        }
        return sid

    def ensure_session(self, session_id: Optional[str], department: str = "public") -> str:
        if session_id and session_id in self._sessions:
            return session_id
        if session_id:
            self._sessions[session_id] = {
                "session_id": session_id,
                "title": "自动创建会话",
                "department": department,
                "history": [],
            }
            return session_id
        return self.create_session(department=department)

    def append_turn(self, session_id: str, role: str, content: str, rewritten_query: Optional[str] = None) -> None:
        sess = self._sessions[session_id]
        sess["history"].append(ChatTurn(role=role, content=content, rewritten_query=rewritten_query))
        max_msgs = self.max_turns * 2
        if len(sess["history"]) > max_msgs:
            sess["history"] = sess["history"][-max_msgs:]

    def get_history(self, session_id: str) -> List[ChatTurn]:
        if session_id not in self._sessions:
            raise KeyError(f"Session {session_id} not found")
        return list(self._sessions[session_id]["history"])

    def list_sessions(self) -> List[Dict[str, Any]]:
        return [
            {
                "session_id": s["session_id"],
                "title": s["title"],
                "department": s["department"],
                "turn_count": len(s["history"]) // 2,
            }
            for s in self._sessions.values()
        ]

    def delete_session(self, session_id: str) -> bool:
        return self._sessions.pop(session_id, None) is not None


class MultiTurnQueryRewriter:
    """
    多轮对话指代消解改写器：
    识别「它/这个/该制度/那…」等指代词与短追问，从上一轮用户问题中提取主题实体进行语义补全。
    """

    PRONOUNS = ("它", "这个", "那个", "该制度", "这项", "上面")
    TOPIC_KEYWORDS = ("年假", "病假", "出差", "住宿", "餐饮", "发布窗口", "周五", "ERR-5042", "VPN", "天枢项目")

    def rewrite(self, history: Sequence[ChatTurn], current_query: str) -> str:
        user_turns = [t for t in history if t.role == "user"]
        if not user_turns:
            return current_query

        last_q = user_turns[-1].rewritten_query or user_turns[-1].content
        # 抽取上一轮的核心业务主题
        anchor = next((kw for kw in self.TOPIC_KEYWORDS if kw in last_q), "")
        if not anchor:
            anchor = re.sub(r"[？?！!。，,请问一下呢吧]", "", last_q)[:10]

        has_pronoun = any(p in current_query for p in self.PRONOUNS)
        is_short_followup = (current_query.startswith("那") or len(current_query.strip()) <= 11) and (
            anchor not in current_query
        )

        if not has_pronoun and not is_short_followup:
            return current_query

        rewritten = current_query
        for p in self.PRONOUNS:
            if p in rewritten:
                rewritten = rewritten.replace(p, anchor)
        if anchor not in rewritten:
            rewritten = f"关于{anchor}，{rewritten}"
        return rewritten


class CitationHighlighter:
    """安全引用溯源高亮器：先 HTML 转义防 XSS，再对命中关键词插入 <mark>...</mark> 高亮标签。"""

    STOP_WORDS = {"怎么", "多少", "什么", "需要", "可以", "请问", "关于", "一下", "是否", "如何", "我们"}

    @classmethod
    def highlight(cls, query: str, snippet: str) -> str:
        safe_text = html.escape(snippet)
        # 提取英文/错误码/数字与长度>=2的中文关键片段
        raw_terms = re.findall(r"[A-Za-z0-9]+(?:[-_][A-Za-z0-9]+)*|[\u4e00-\u9fff]{2,4}", query)
        candidates: List[str] = []
        for term in raw_terms:
            if term in cls.STOP_WORDS:
                continue
            if len(term) > 2 and re.fullmatch(r"[\u4e00-\u9fff]+", term):
                # 同时加入完整短语和二元词组以提高命中率
                candidates.append(term)
                for i in range(len(term) - 1):
                    bg = term[i : i + 2]
                    if bg not in cls.STOP_WORDS:
                        candidates.append(bg)
            else:
                candidates.append(term)

        # 去重并按长度降序，优先匹配长词避免碎片化高亮
        unique_terms = sorted(set(candidates), key=len, reverse=True)
        matched_terms = [t for t in unique_terms if t.lower() in safe_text.lower()][:6]
        if not matched_terms:
            return safe_text

        pattern = re.compile("|".join(re.escape(t) for t in matched_terms), flags=re.IGNORECASE)
        return pattern.sub(lambda m: f"<mark>{m.group(0)}</mark>", safe_text)


class StreamingKBService:
    """集多会话、多轮改写、混合检索、引用高亮与 SSE 流式生成于一体的项目二完整服务。"""

    def __init__(self) -> None:
        self.sessions = SessionManager(max_turns=6)
        self.rewriter = MultiTurnQueryRewriter()
        self.docs: List[Dict[str, str]] = [
            {
                "doc_id": "hr-leave",
                "source": "HR员工休假手册.md",
                "section": "年假与审批流程",
                "department": "public",
                "content": "入职满1年员工享有5天带薪年假，年假需提前3个工作日在OA系统提交审批；病假超过2天须提供公立医院诊断证明。",
            },
            {
                "doc_id": "fin-travel",
                "source": "财务差旅制度.md",
                "section": "差旅住宿与餐补标准",
                "department": "public",
                "content": "一线城市（北京、上海、深圳、广州）出差酒店住宿上限每晚600元，餐饮补助按每天100元定额随工资发放，出差需提前3天在OA发起申请。",
            },
            {
                "doc_id": "ops-5042",
                "source": "运维排障手册.md",
                "section": "ERR-5042网关超时",
                "department": "rd",
                "content": "错误码ERR-5042表示星辰统一网关调用上游推理服务超时（默认15秒），排查步骤：检查上游显存水位并调大proxy_read_timeout。",
            },
        ]

    def _retrieve(self, query: str, department: str = "public", top_k: int = 2) -> List[Tuple[Dict[str, str], float]]:
        allowed = {"public", department}
        q_lower = query.lower()
        q_terms = set(re.findall(r"[a-z0-9]+(?:[-_][a-z0-9]+)*", q_lower))
        for chunk in re.findall(r"[\u4e00-\u9fff]+", q_lower):
            for i in range(len(chunk) - 1):
                q_terms.add(chunk[i : i + 2])

        q_codes = set(re.findall(r"[a-z0-9]+[-_][a-z0-9]+", q_lower))
        scored: List[Tuple[Dict[str, str], float]] = []
        for d in self.docs:
            if d["department"] not in allowed:
                continue
            text = (d["section"] + " " + d["content"]).lower()
            hits = sum(1 for t in q_terms if t in text)
            code_bonus = 0.5 if (q_codes and any(c in text for c in q_codes)) else 0.0
            score = round(min(1.0, hits / max(1, len(q_terms)) + code_bonus), 4)
            if score >= 0.18:
                scored.append((d, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def prepare_answer(
        self,
        session_id: Optional[str],
        question: str,
        department: str = "public",
    ) -> Dict[str, Any]:
        t0 = time.perf_counter()
        sid = self.sessions.ensure_session(session_id, department=department)
        history = self.sessions.get_history(sid)
        rewritten = self.rewriter.rewrite(history, question)
        hits = self._retrieve(rewritten, department=department, top_k=2)

        if not hits:
            answer = "抱歉，星辰知识库中未检索到足够依据回答当前问题。"
            citations: List[HighlightedCitation] = []
        else:
            citations = [
                HighlightedCitation(
                    index=i,
                    doc_id=d["doc_id"],
                    source=d["source"],
                    section=d["section"],
                    raw_snippet=d["content"],
                    highlighted_snippet=CitationHighlighter.highlight(rewritten, d["content"]),
                    score=sc,
                )
                for i, (d, sc) in enumerate(hits, start=1)
            ]
            top_c = citations[0]
            answer = f"根据《{top_c.source}》（{top_c.section}）：{top_c.raw_snippet} [1]"

        # 将本轮用户原问题与回答写入会话记忆
        self.sessions.append_turn(sid, "user", question, rewritten_query=rewritten)
        self.sessions.append_turn(sid, "assistant", answer)

        return {
            "session_id": sid,
            "original_question": question,
            "rewritten_query": rewritten,
            "answer": answer,
            "citations": citations,
            "latency_ms": round((time.perf_counter() - t0) * 1000.0, 2),
        }

    def stream_sse_events(
        self,
        session_id: Optional[str],
        question: str,
        department: str = "public",
        chunk_chars: int = 12,
    ) -> Generator[str, None, None]:
        """按标准 SSE 帧协议依次流式推送：meta -> citation -> token -> done。"""
        result = self.prepare_answer(session_id=session_id, question=question, department=department)

        # 1. 元数据帧（前端可展示“已理解上下文：xxx”）
        meta_payload = {
            "session_id": result["session_id"],
            "rewritten_query": result["rewritten_query"],
        }
        yield f"event: meta\ndata: {json.dumps(meta_payload, ensure_ascii=False)}\n\n"

        # 2. 高亮引用帧（Citations-First，右侧溯源栏瞬间点亮）
        for c in result["citations"]:
            cit_payload = {
                "index": c.index,
                "doc_id": c.doc_id,
                "source": c.source,
                "section": c.section,
                "highlighted_snippet": c.highlighted_snippet,
                "score": c.score,
            }
            yield f"event: citation\ndata: {json.dumps(cit_payload, ensure_ascii=False)}\n\n"

        # 3. 正文打字机分片帧
        answer = result["answer"]
        for i in range(0, len(answer), chunk_chars):
            piece = answer[i : i + chunk_chars]
            yield f"event: token\ndata: {json.dumps({'delta': piece}, ensure_ascii=False)}\n\n"

        # 4. 完成结束帧
        done_payload = {
            "session_id": result["session_id"],
            "total_chars": len(answer),
            "latency_ms": result["latency_ms"],
        }
        yield f"event: done\ndata: {json.dumps(done_payload, ensure_ascii=False)}\n\n"


if __name__ == "__main__":
    svc = StreamingKBService()
    sid = svc.sessions.create_session("年假咨询")
    r1 = svc.prepare_answer(sid, "入职满一年年假有几天？")
    r2 = svc.prepare_answer(sid, "它需要提前几天OA申请？")
    print("第2轮改写:", r2["rewritten_query"])
    print("高亮引用:", r2["citations"][0].highlighted_snippet)


### 🔍 步骤 1 代码深度解析

1. `CitationHighlighter.highlight` 先调用 `html.escape(snippet)` 转义原文本，再一次性用合并正则替换所有非重叠命中词为 `<mark>...</mark>`，既安全防 XSS 又彻底避免嵌套标签问题。
2. `stream_sse_events` 严格遵循 `event: <type>\ndata: <json>\n\n` SSE 标准格式，前端用原生 `fetch + ReadableStream` 即可零依赖实现丝滑打字机与溯源卡片联动。

**预期运行输出参考：**
```text
第2轮改写: 年假需要提前几天OA申请？
高亮引用: 入职满1年员工享有5天带薪<mark>年假</mark>，<mark>年假</mark>需<mark>提前</mark>3个工作日在<mark>OA</mark>系统提交审批；病假超过2天须提供公立医院诊断证明。
```


## 💻 实操步骤 2：步骤二：实现多会话管理与 SSE 流式问答 FastAPI 服务（day37_sse_api.py）

- **目标文件**：`versions/day37/src/day37_sse_api.py`
- **老陈代码旁白**：现在把 `StreamingKBService` 挂载到 FastAPI，完整暴露会话 CRUD、同步问答和 `/api/v1/chat/stream` SSE 流式接口，并设置防 Nginx 缓冲的关键响应头！


In [ ]:
"""
Day 37 - 步骤二：阶段项目二(下) 多会话管理与 SSE 流式问答 FastAPI 后端服务
"""

from __future__ import annotations

import sys
from pathlib import Path
from typing import Any, Dict, List, Optional

from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from pydantic import BaseModel, Field

ROOT = Path(__file__).resolve().parents[1]
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.day37_session_highlight import StreamingKBService


class CreateSessionRequest(BaseModel):
    title: str = Field(default="新对话", max_length=64)
    department: str = Field(default="public")


class ChatRequest(BaseModel):
    session_id: Optional[str] = None
    question: str = Field(..., min_length=2, max_length=500)
    department: str = Field(default="public")


def create_chat_app(service: Optional[StreamingKBService] = None) -> FastAPI:
    svc = service if service is not None else StreamingKBService()
    app = FastAPI(
        title="StarMind-KB v2.0 多轮流式问答服务",
        version="2.1.0",
    )

    @app.post("/api/v1/sessions")
    def create_session(req: CreateSessionRequest) -> Dict[str, str]:
        sid = svc.sessions.create_session(title=req.title, department=req.department)
        return {"session_id": sid, "title": req.title, "department": req.department}

    @app.get("/api/v1/sessions")
    def list_sessions() -> Dict[str, Any]:
        items = svc.sessions.list_sessions()
        return {"total": len(items), "items": items}

    @app.get("/api/v1/sessions/{session_id}/history")
    def get_session_history(session_id: str) -> Dict[str, Any]:
        try:
            turns = svc.sessions.get_history(session_id)
        except KeyError as exc:
            raise HTTPException(status_code=404, detail=str(exc)) from exc
        return {
            "session_id": session_id,
            "history": [
                {"role": t.role, "content": t.content, "rewritten_query": t.rewritten_query}
                for t in turns
            ],
        }

    @app.delete("/api/v1/sessions/{session_id}")
    def delete_session(session_id: str) -> Dict[str, Any]:
        if not svc.sessions.delete_session(session_id):
            raise HTTPException(status_code=404, detail="Session not found")
        return {"deleted": True, "session_id": session_id}

    @app.post("/api/v1/chat/ask")
    def chat_ask(req: ChatRequest) -> Dict[str, Any]:
        res = svc.prepare_answer(
            session_id=req.session_id,
            question=req.question,
            department=req.department,
        )
        return {
            "session_id": res["session_id"],
            "rewritten_query": res["rewritten_query"],
            "answer": res["answer"],
            "latency_ms": res["latency_ms"],
            "citations": [
                {
                    "index": c.index,
                    "doc_id": c.doc_id,
                    "source": c.source,
                    "section": c.section,
                    "highlighted_snippet": c.highlighted_snippet,
                    "score": c.score,
                }
                for c in res["citations"]
            ],
        }

    @app.post("/api/v1/chat/stream")
    def chat_stream(req: ChatRequest) -> StreamingResponse:
        # 生产级 SSE 必配响应头：禁止代理缓存与 Nginx 缓冲
        headers = {
            "Cache-Control": "no-cache",
            "Connection": "keep-alive",
            "X-Accel-Buffering": "no",
        }
        return StreamingResponse(
            svc.stream_sse_events(
                session_id=req.session_id,
                question=req.question,
                department=req.department,
            ),
            media_type="text/event-stream",
            headers=headers,
        )

    return app


app = create_chat_app()

if __name__ == "__main__":
    from fastapi.testclient import TestClient

    client = TestClient(create_chat_app())
    sid = client.post("/api/v1/sessions", json={"title": "差旅咨询"}).json()["session_id"]
    client.post("/api/v1/chat/ask", json={"session_id": sid, "question": "北京出差住酒店上限是多少？"})
    with client.stream("POST", "/api/v1/chat/stream", json={"session_id": sid, "question": "那餐饮补助呢？"}) as r:
        print("SSE Content-Type:", r.headers["content-type"])
        for line in r.iter_lines():
            if line:
                print(" ", line)


### 🔍 步骤 2 代码深度解析

1. `/api/v1/chat/stream` 返回 `StreamingResponse(..., media_type='text/event-stream', headers={'X-Accel-Buffering': 'no', ...})`，确保经过 Nginx 也能实时逐帧到达浏览器。
2. 会话 CRUD 与流式问答共享同一个 `StreamingKBService` 实例，调用完流式接口后立刻查询 `/api/v1/sessions/{sid}/history` 就能看到完整持久化的对话记录。

**预期运行输出参考：**
```text
SSE Content-Type: text/event-stream; charset=utf-8
  event: meta
  data: {"session_id": "sess_...", "rewritten_query": "关于出差，那餐饮补助呢？"}
  event: citation
  data: {"index": 1, ..., "highlighted_snippet": "...<mark>餐饮</mark>补助按每天100元..."}
  event: token
  data: {"delta": "根据《财务差旅制度.md"}
  ...
  event: done
  data: {"session_id": "sess_...", "total_chars": ..., "latency_ms": ...}
```


## 🧪 当日自动化单元测试

在终端运行 `pytest versions/day37/tests/test_day37.py -v` 即可验收全部测试断言。下面展示完整测试脚本源码，供对照学习测试写法：

```python
"""Day 37 全链路端到端集成测试：验证多会话隔离、多轮指代改写、安全高亮与 SSE 流式解析。"""

import json
from typing import Any, List, Tuple

from fastapi.testclient import TestClient

from src.day37_session_highlight import CitationHighlighter
from src.day37_sse_api import create_chat_app


def _parse_sse_frames(raw_text: str) -> List[Tuple[str, Any]]:
    frames: List[Tuple[str, Any]] = []
    for block in raw_text.strip().split("\n\n"):
        event_type = "message"
        data_str = ""
        for line in block.splitlines():
            if line.startswith("event:"):
                event_type = line.split(":", 1)[1].strip()
            elif line.startswith("data:"):
                data_str = line.split(":", 1)[1].strip()
        if data_str:
            frames.append((event_type, json.loads(data_str)))
    return frames


def test_citation_highlighter_prevents_xss_and_marks_keywords():
    raw = "年假需提前3天OA申请，<script>alert(1)</script>严禁绕过审批。"
    out = CitationHighlighter.highlight("年假提前OA申请", raw)
    assert "<script>" not in out
    assert "&lt;script&gt;" in out
    assert "<mark>年假</mark>" in out
    assert "<mark>OA</mark>" in out


def test_multi_session_isolation_and_multiturn_rewrite():
    client = TestClient(create_chat_app())

    sid_a = client.post("/api/v1/sessions", json={"title": "HR休假"}).json()["session_id"]
    sid_b = client.post("/api/v1/sessions", json={"title": "运维排障", "department": "rd"}).json()["session_id"]

    # Session A 第一轮聊年假
    r_a1 = client.post("/api/v1/chat/ask", json={"session_id": sid_a, "question": "入职满一年年假有几天？"})
    assert r_a1.status_code == 200
    assert "5天" in r_a1.json()["answer"]

    # Session B 第一轮聊 ERR-5042
    r_b1 = client.post(
        "/api/v1/chat/ask",
        json={"session_id": sid_b, "question": "ERR-5042 报错原因是什么？", "department": "rd"},
    )
    assert "超时" in r_b1.json()["answer"]

    # Session A 第二轮追问使用指代词「它」，应准确锚定到「年假」而非 Session B 的 ERR-5042
    r_a2 = client.post("/api/v1/chat/ask", json={"session_id": sid_a, "question": "它需要提前几天OA申请？"})
    body_a2 = r_a2.json()
    assert "年假" in body_a2["rewritten_query"]
    assert "ERR-5042" not in body_a2["rewritten_query"]
    assert "<mark>" in body_a2["citations"][0]["highlighted_snippet"]

    # 验证历史隔离：A 与 B 各自只有自己的记录
    hist_a = client.get(f"/api/v1/sessions/{sid_a}/history").json()["history"]
    hist_b = client.get(f"/api/v1/sessions/{sid_b}/history").json()["history"]
    assert len(hist_a) == 4  # 2 turns * 2 messages
    assert len(hist_b) == 2  # 1 turn * 2 messages


def test_sse_streaming_endpoint_full_protocol():
    client = TestClient(create_chat_app())
    sid = client.post("/api/v1/sessions", json={"title": "差旅SSE测试"}).json()["session_id"]

    # 第一轮同步建立上下文
    client.post("/api/v1/chat/ask", json={"session_id": sid, "question": "北京出差住酒店上限是多少？"})

    # 第二轮走 SSE 流式接口追问
    resp = client.post("/api/v1/chat/stream", json={"session_id": sid, "question": "那餐饮补助是多少？"})
    assert resp.status_code == 200
    assert "text/event-stream" in resp.headers["content-type"]
    assert resp.headers.get("x-accel-buffering") == "no"

    frames = _parse_sse_frames(resp.text)
    event_types = [ev for ev, _ in frames]
    assert event_types[0] == "meta"
    assert "citation" in event_types
    assert "token" in event_types
    assert event_types[-1] == "done"

    meta = frames[0][1]
    assert "出差" in meta["rewritten_query"]

    reconstructed = "".join(payload["delta"] for ev, payload in frames if ev == "token")
    assert "100元" in reconstructed

    # 确认流式回答已持久化进入历史
    hist = client.get(f"/api/v1/sessions/{sid}/history").json()["history"]
    assert len(hist) == 4
    assert hist[-1]["content"] == reconstructed

```


## 📝 课后梯度实战作业与标准答案

建议先在下方新建空白单元格独立完成编码，运行通过后再核对参考答案。


### 作业 1（基础巩固）：作业一：实现客户端 SSE 流式帧解析器（parse_sse_stream）

**题目要求：**
编写通用 `parse_sse_lines(lines)` 函数，将多行 SSE 文本流解析为 `List[Dict]`（每条包含 `event` 和 `data` 字典），并兼容空行与注释心跳行（以 `:` 开头的 `: ping` 保活帧）。

**思路提示：** 逐行扫描：跳过以 `:` 开头的注释行；遇到空行时提交当前缓冲的事件对象并重置。


In [ ]:
"""Day 37 作业一参考答案：兼容心跳帧的 SSE 客户端解析器"""
import json
from typing import Any, Dict, Iterable, List


def parse_sse_lines(lines: Iterable[str]) -> List[Dict[str, Any]]:
    events: List[Dict[str, Any]] = []
    cur_event = "message"
    cur_data: List[str] = []

    for raw in lines:
        line = raw.rstrip("\r\n")
        if line.startswith(":"):
            continue  # 跳过 SSE 心跳保活注释帧
        if line == "":
            if cur_data:
                events.append({"event": cur_event, "data": json.loads("\n".join(cur_data))})
                cur_event = "message"
                cur_data = []
            continue
        if line.startswith("event:"):
            cur_event = line.split(":", 1)[1].strip()
        elif line.startswith("data:"):
            cur_data.append(line.split(":", 1)[1].strip())

    if cur_data:
        events.append({"event": cur_event, "data": json.loads("\n".join(cur_data))})
    return events


if __name__ == "__main__":
    sample = [
        ": keep-alive",
        "event: token",
        'data: {"delta": "你好"}',
        "",
        "event: done",
        'data: {"ok": true}',
        "",
    ]
    parsed = parse_sse_lines(sample)
    print(parsed)
    assert len(parsed) == 2 and parsed[0]["data"]["delta"] == "你好"


**答案解析与评分标准：**

生产网关通常每 15~30 秒发一次 `: ping` SSE 注释帧防止连接超时断开，客户端解析器必须优雅忽略注释帧。


### 作业 2（企业实战）：作业二：实现基于滑动 Token 预算的对话历史裁剪器

**题目要求：**
相比按轮数截断，真实 LLM 更关心字符/Token 预算。编写 `trim_history_by_budget(turns, max_chars=80)`，从最新一轮倒序保留对话记录，确保总字符数不超过 `max_chars` 且保持原有正序。

**思路提示：** 逆序遍历 `reversed(turns)` 累加字符数，超限时 `break`，最后再 `reverse()` 恢复时间顺序。


In [ ]:
"""Day 37 作业二参考答案：基于字符/Token 预算的滑动历史裁剪器"""
from typing import Dict, List, Sequence


def trim_history_by_budget(turns: Sequence[Dict[str, str]], max_chars: int = 80) -> List[Dict[str, str]]:
    kept: List[Dict[str, str]] = []
    used = 0
    for item in reversed(turns):
        size = len(item["content"])
        if used + size > max_chars and kept:
            break
        kept.append(item)
        used += size
    kept.reverse()
    return kept


if __name__ == "__main__":
    hist = [
        {"role": "user", "content": "第一轮很长的历史提问内容" * 3},
        {"role": "assistant", "content": "第一轮很长的历史回答内容" * 3},
        {"role": "user", "content": "北京住宿上限多少？"},
        {"role": "assistant", "content": "每晚600元。"},
    ]
    trimmed = trim_history_by_budget(hist, max_chars=30)
    print("裁剪后保留:", trimmed)
    assert len(trimmed) == 2 and trimmed[0]["content"] == "北京住宿上限多少？"


**答案解析与评分标准：**

按 Token 预算裁剪比固定按轮数裁剪更安全，避免某一条超长日志粘贴瞬间撑爆 LLM 上下文窗口。


### 作业 3（架构进阶）：作业三：实现前端引用角标 `[1][2]` 自动校验与死链清理器

**题目要求：**
大模型偶尔会产生「幽灵引用」（例如只给了 2 条引用，回答里却瞎写了个 `[3]`）。请实现 `sanitize_citation_markers(answer, max_index)`：自动移除所有超过 `max_index` 或等于 `0` 的非法角标，仅保留合法引用角标。

**思路提示：** 用 `re.sub(r'\[(\d+)\]', repl, answer)` 检查捕获数字是否在 `1 <= num <= max_index`。


In [ ]:
"""Day 37 作业三参考答案：大模型幽灵引用角标校验与清理器"""
import re


def sanitize_citation_markers(answer: str, max_index: int) -> str:
    def _repl(match: re.Match) -> str:
        idx = int(match.group(1))
        return match.group(0) if 1 <= idx <= max_index else ""

    return re.sub(r"\[(\d+)\]", _repl, answer)


if __name__ == "__main__":
    raw = "北京住宿上限600元[1]，餐补100元[2]，且周末发红包[5][0]。"
    clean = sanitize_citation_markers(raw, max_index=2)
    print("清理后:", clean)
    assert "[1]" in clean and "[2]" in clean
    assert "[5]" not in clean and "[0]" not in clean


**答案解析与评分标准：**

幽灵角标会导致前端点击引用跳转直接报错或 404，加上这一行正则兜底能彻底守住产品体验底线。


---

## 🎯 老陈收尾复盘

绝了！从多会话创建、第一轮提问、第二轮追问自动改写、证据 `<mark>` 高亮，到 SSE 打字机逐帧推送与历史落库，整个 StarMind-KB v2.0 全链路已经 100% 闭环通过！今晚大家早点休息，明天（Day 38）上台做项目二技术答辩与并发性能压测重构！
